# Calculate corn evapotranspiration and irrigation with `pyfao56`

Convert hourly precipitation and grass-reference evapotranspiration
($ET_0$) into daily agricultural water-balance forcing and apply the FAO-56
dual crop-coefficient method for corn.

The dual-coefficient method partitions actual evapotranspiration into:

$$ ET_a = T + E $$

where

$$ T = K_s K_{cb} ET_0 $$

is water-stressed crop transpiration and

$$ E = K_e ET_0 $$

is bare soil evaporation.

During fallow periods, the basal crop coefficient and crop cover are set to zero.
This eliminates crop transpiration while retaining rainfall-driven bare-soil
evaporation.

In [ ]:
import s3fs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfao56 as fao
from tqdm.notebook import tqdm

# Set up S3 access
s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

# Load in hourly reference ET and precipitation for each site
ref_et = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/ref_et.csv', index_col=0, parse_dates=True)
precip = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/precip.csv', index_col=0, parse_dates=True)

all_sites = ref_et.columns

Because `pyfao56` uses a daily water balance, we'll have to sum hourly values daily
totals, then redistribute to hourly values at the end.

In [ ]:
# Check that the inputs are nonnegative
assert not (precip < 0).any().any(), "Negative precipitation values found."
assert not (ref_et < 0).any().any(), "Negative ET0 values found."

# Check that the inputs do not contain any NaN
assert not precip.isna().any().any(), "NaN precipitation values found."
assert not ref_et.isna().any().any(), "NaN ET0 values found."

# Aggregate hourly totals to daily totals
ppt_daily = precip.resample("D").sum(min_count=24)
eto_daily = ref_et.resample("D").sum(min_count=24)

# Check that the daily totals do not contain any NaN
assert not ppt_daily.isna().any().any(), "NaN precipitation values found."
assert not eto_daily.isna().any().any(), "NaN ET0 values found."

## Define corn, soil and irrigation assumptions

A separate `pyfao56` calculation is run for each site and calendar year.

For each simulation:

1. Daily precipitation and $ET_0$ are supplied through a `Weather` object.
2. Corn growth is specified using four growth stages.
3. A daily `Update` object specifies basal crop coefficient, crop height, and
   fractional cover.
4. Before planting and after harvest:
   - $K_{cb}=0$
   - crop height = 0
   - crop cover = 0.05

   Therefore, transpiration is zero during fallow periods, but soil evaporation
   can continue after rainfall.
5. Automatic irrigation is allowed only between planting and harvest.
6. For simplicity, we'll assume corn is planted on April 15

In [ ]:
## Define key parameters for FAO-56 calculations for maize (field corn)
# Assume planing on April 15
planting_day = '04-15'

# Growth stages in days, from FAO-56 Table 11 for Maize (grain) for Idaho
growth_stages = {
    'Lini_crop': 30,
    'Ldev': 40,
    'Lmid': 50,
    'Lend': 50,
}

# Basal crop coefficients for Maize from FAO-56 Table 17
kcb_values = {
    'Kcb_ini': 0.15,
    'Kcb_mid': 1.15,
    'Kcb_end': 0.15,
}

# Crop height and coverage parameters
h_ini_m = 0.05  # Initial crop height
h_max_m = 2.0  # Maximum height from FAO-56 Table 12
fc_init = 0.10  # Initial crop cover (fractional land area)
fc_max = 0.95  # Maximum crop cover

# Rooting depth, from FAO-56 Table 22
Zr_ini_m = 0.15  # Initial rooting depth
Zr_max_m = 1.35  # Maximum rooting depth in meters (range is 1.0-1.7 m)
p_base = 0.55  # Soil water depletion fraction for no stress (see p. 162 for description)

# Surface evaporation layer
ze_m = 0.10  # Depth of surface evaporation layer in meters, 0.10-0.15 from FAO-56 p. 144
rew_mm = 9.0  # total stage 1 evaporation (mm) from FAO-56 Table 19

In [ ]:
from byte_util import van_genuchten_vwc

# Calculate "average" field capacity and wilting point across all sites
soil_params_path = (f'{s3_base_path}/'
                    'input-data/processed-data/soil_physical_parameters.parquet')
soil_params = pd.read_parquet(soil_params_path)
all_sites = list(soil_params.index.get_level_values(0).unique())

fc = np.empty(len(all_sites*2,), dtype=float)
wp = np.empty(len(all_sites*2,), dtype=float)

i = 0
for site in all_sites:
    for horizon in ['A', 'B']:
        theta_r = soil_params.loc[(site, horizon), 'theta_r']
        theta_s = soil_params.loc[(site, horizon), 'theta_s']
        alpha = soil_params.loc[(site, horizon), 'alpha_m']
        n = soil_params.loc[(site, horizon), 'n']

        # Define field capacity as water content at -3.3 m
        fc[i] = van_genuchten_vwc(-3.3, theta_r, theta_s, alpha, n)

        # Define wilting point as water content at -15 m
        wp[i] = van_genuchten_vwc(-15, theta_r, theta_s, alpha, n)

        i += 1

# Assume homogeneous soil
field_capacity = np.nanmean(fc)
wilting_point = np.nanmean(wp)
theta_init = field_capacity  # initial water content

In [ ]:
# Calculate daily Kcb, crop height, and fractional crop cover values
all_dates = pd.date_range(ppt_daily.index[0], ppt_daily.index[-1], freq='D')
crop_curves = pd.DataFrame(columns=['Kcb', 'h', 'fc'], index=all_dates)
crop_curves.loc[:, :] = 0  # 0 outside of growing season
crop_curves['fc'] = 0.05  # 5% fractional cover during fallow period (from corn residue, etc.)

for year in all_dates.year.unique():
    dates = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")

    # Assume planting date on April 15
    plant_date = pd.Timestamp(f'{year}-{planting_day}')

    d0 = plant_date
    d1 = d0 + pd.Timedelta(days=growth_stages['Lini_crop'])
    d2 = d1 + pd.Timedelta(days=growth_stages['Ldev'])
    d3 = d2 + pd.Timedelta(days=growth_stages['Lmid'])
    d4 = d3 + pd.Timedelta(days=growth_stages['Lend'])

    # Initial stage
    mask = (all_dates >= d0) & (all_dates < d1)
    crop_curves.loc[mask, "Kcb"] = kcb_values['Kcb_ini']
    crop_curves.loc[mask, "h"] = h_ini_m
    crop_curves.loc[mask, "fc"] = fc_init

    # Development stage
    mask = (all_dates >= d1) & (all_dates < d2)
    n = mask.sum()
    crop_curves.loc[mask, "Kcb"] = np.linspace(kcb_values['Kcb_ini'], kcb_values['Kcb_mid'], n, endpoint=False)
    crop_curves.loc[mask, "h"] = np.linspace(h_ini_m, h_max_m, n, endpoint=False)
    crop_curves.loc[mask, "fc"] = np.linspace(fc_init, fc_max, n, endpoint=False)

    # Mid-season stage
    mask = (all_dates >= d2) & (all_dates < d3)
    crop_curves.loc[mask, "Kcb"] = kcb_values['Kcb_mid']
    crop_curves.loc[mask, "h"] = h_max_m
    crop_curves.loc[mask, "fc"] = fc_max

    # Late-season stage
    mask = (all_dates >= d3) & (all_dates < d4)
    n = mask.sum()
    crop_curves.loc[mask, "Kcb"] = np.linspace(kcb_values['Kcb_mid'], kcb_values['Kcb_end'], n, endpoint=False)
    crop_curves.loc[mask, "h"] = h_max_m
    crop_curves.loc[mask, "fc"] = fc_max

In [ ]:
# Plot one example crop curve
fig, ax = plt.subplots()

ax.plot(crop_curves.index, crop_curves["Kcb"], label="Kcb")
ax.plot(crop_curves.index, crop_curves["h"], label="Crop height (m)")
ax.plot(crop_curves.index, crop_curves["fc"], label="Crop cover (fraction)")
ax.set(ylabel='Kcb, height (m) or fractional cover',
       xlim=pd.to_datetime(['2010-01-01', '2011-12-31']))
ax.legend()

### Create the `pyfao56` weather object

The `Weather` object normally contains meteorological variables used to
calculate reference ET, but since we've already calculated $ET_0$, we'll supply `ETref` directly. Same for `Rain`.

In [ ]:
# Create pyfao56 Weather objects for each year
# Note that we need to do this for each year because of leap years
# We'll also do this for each site, since weather data is variable across sites
wth_obj_all_sites = {}
for site in all_sites:
    wth_objects = {}
    for year in all_dates.year.unique():
        start_date = f'{year}-01-01'
        end_date = f'{year}-12-31'
        wth = fao.Weather()

        # Short grass reference ET
        wth.rfcrp = "S"

        # So station metadata are not used
        wth.z = np.nan
        wth.lat = np.nan
        wth.wndht = 2.0

        wdata = pd.DataFrame(index=ppt_daily.loc[start_date:end_date, :].index.strftime("%Y-%j"))

        for column in wth.cnames:
            wdata[column] = np.nan

        wdata["Rain"] = ppt_daily.loc[start_date:end_date, site].values
        wdata["ETref"] = eto_daily.loc[start_date:end_date, site].values

        wth.wdata = wdata[wth.cnames]
        wth_objects[year] = wth
    wth_obj_all_sites[site] = wth_objects

In [ ]:
# Create pyfao56 Parameters and Update objects for each year
# Note that we need to do this for each year because of leap years
par_objects = {}
upd_objects = {}
for year in all_dates.year.unique():
    par = fao.Parameters()

    planting_day_of_year = pd.to_datetime(f'{year}-{planting_day}').dayofyear

    # Include the preplant fallow interval in the model's initial stage.
    par.Lini = planting_day_of_year - 1 + growth_stages["Lini_crop"]
    par.Ldev = growth_stages["Ldev"]
    par.Lmid = growth_stages["Lmid"]
    par.Lend = growth_stages["Lend"]

    # Single coefficients are included because Parameters requires them, although they're unused
    par.Kcmini = 0.30
    par.Kcmmid = 1.20
    par.Kcmend = 0.35

    par.Kcbini = kcb_values["Kcb_ini"]
    par.Kcbmid = kcb_values["Kcb_mid"]
    par.Kcbend = kcb_values["Kcb_end"]

    par.hini = h_ini_m
    par.hmax = h_max_m

    par.thetaFC = field_capacity
    par.thetaWP = wilting_point
    par.theta0 = theta_init

    par.Zrini = Zr_ini_m
    par.Zrmax = Zr_max_m
    par.pbase = p_base

    par.Ze = ze_m
    par.REW = rew_mm

    par_objects[year] = par

    # Also make pyfao56 update object from annual crop_curves
    upd = fao.Update()
    one_year_curves = crop_curves.loc[f'{year}-01-01':f'{year}-12-31', :]
    one_year_curves.index = one_year_curves.index.strftime("%Y-%j")
    upd.udata = one_year_curves[['Kcb', 'h', 'fc']].copy()
    upd_objects[year] = upd

### Configure automatic irrigation

Turn on irrigation between planting and harvest. The management-allowed depletion (`mad`) controls when irrigation is triggered, which is commonly 50% for corn.

In [ ]:
# Irrigate when root-zone depletion reaches 50% of total available water
management_allowed_depletion = 0.50

irr_objects = {}
for year in all_dates.year.unique():
    planting_date = pd.to_datetime(f'{year}-{planting_day}')
    growing_length = sum(growth_stages.values())
    harvest_date = planting_date + pd.Timedelta(days=growing_length)
    autoirr = fao.AutoIrrigate()

    autoirr.addset(planting_date.strftime("%Y-%j"),
                   harvest_date.strftime("%Y-%j"),
                   mad=management_allowed_depletion)
    irr_objects[year] = autoirr

### Next, calculate FAO-56 outputs for each site and year

The principal outputs retained for MIN3P forcing are:

- `Rain`: precipitation
- `Irrig`: applied irrigation
- `E`: bare-soil evaporation
- `T`: actual crop transpiration
- `ETa`: total actual evapotranspiration
- `Kcb`, `Ke`, and `Ks`: diagnostic coefficients
- `Dr`: root-zone soil-water depletion

Because `Kcb` and crop cover are zero outside the growing season, `T` should
be zero during fallow periods. However, `E` can be positive after rainfall,
including before planting and after harvest.

In [ ]:
## Finally, run FAO-56 calculations for all sites and years, and combine outputs into a single dataframe per site
recalculate = False
all_fao_output = {}

# Only save a subset of columns per site
output_columns = ['ETref', 'Kcm', 'ETcm', 'tKcb', 'Kcb', 'ETcb', 'h', 'Kcmax', 'ETmax', 'fc', 'fw', 'few', 'De', 'Kr',
                  'Ke', 'E', 'DPe', 'Kc', 'ETc', 'TAW', 'TAWrmax', 'TAWb', 'Zr', 'p', 'RAW', 'Ks', 'Ka', 'ETa', 'T',
                  'DP', 'Dinc', 'Dr', 'fDr', 'Drmax', 'fDrmax', 'Db', 'fDb', 'Ksend', 'Irrig', 'IrrLoss', 'Rain', 'Runoff',]

for site in tqdm(all_sites, desc='Iterating over sites...'):
    if recalculate:
        fao_output = pd.DataFrame(index=all_dates, columns=output_columns)
        wth_objects = wth_obj_all_sites[site]
        for year in tqdm(all_dates.year.unique(), desc='Iterating through each year...', leave=False):
            start_date = pd.Timestamp(f"{year}-01-01").strftime("%Y-%j")
            end_date = pd.Timestamp(f"{year}-12-31").strftime("%Y-%j")

            # Get weather, parameter, update and irrigation objects
            wth = wth_objects[year]
            par = par_objects[year]
            upd = upd_objects[year]
            autoirr = irr_objects[year]

            # Note that cons_p assumes that soil water depletion fraction for no stress (p_base) is constant
            # Figure 41 in FAO-56 shows the impact of constant vs time-varying p value
            mdl = fao.Model(start_date, end_date, par, wth, upd=upd, autoirr=autoirr, cons_p=True)
            mdl.run()

            out = mdl.odata.copy()

            # Convert the yyyy-ddd index back to a DatetimeIndex
            out.index = pd.to_datetime(out.index, format="%Y-%j")
            fao_output.loc[out.index, output_columns] = out[output_columns]

        fao_output.to_csv(f'{s3_base_path}/input-data/processed-data/climate/{site}_FAOWaterBalance.csv')
    else:
        fao_output = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/{site}_FAOWaterBalance.csv', index_col=0, parse_dates=True)

    all_fao_output[site] = fao_output

## Plot two years of daily data for each site

In [ ]:
## Plot two random years per site to make sure they look reasonable
for site in all_sites:
    test_year = 2013
    fao_output = all_fao_output[site]

    sub_df = fao_output.loc[f'{test_year}-01-01':f'{test_year+1}-12-31', :]
    fig, ax = plt.subplots(2, figsize=(10, 6), sharex=True, tight_layout=True)

    # Plot evapotranspiration
    ax[0].plot(sub_df.index, sub_df['E'], color='saddlebrown', label='Daily soil water evaporation (mm)')
    ax[0].plot(sub_df.index, sub_df['T'], color='forestgreen', label='Actual plant transpiration (mm)')

    # Add in precipitation and irrigation as stacked bar chart from the top
    ax[1].bar(sub_df.index, sub_df['Rain'], color='steelblue', label='Precipitation')
    ax[1].bar(sub_df.index, sub_df['Irrig'], color='firebrick', label='Irrigation')

    # Add in growing season
    for i in range(2):
        planting_date = pd.to_datetime(f'{test_year+i}-{planting_day}')
        harvest_date = planting_date + pd.Timedelta(days=growing_length)
        for j in range(2):
            ax[j].axvspan(planting_date, harvest_date, alpha=0.1, color='k')

    ax[0].set(ylabel='Daily evapotranspiration (mm)', title=site)
    ax[1].set(ylabel='Daily water addition (mm)', xlim=[sub_df.index[0], sub_df.index[-1]])
    ax[0].legend()
    ax[1].legend()

    fig.savefig(f'plots/{site}_ExampleIrrigation.png', dpi=300, bbox_inches='tight')

    if site != 'Yolo':
        plt.close(fig)

# Extra checks and calculations

## Check irrigation frequency and magnitude

In [ ]:
# Combine selected daily outputs across sites
daily_outputs = []

for site, df in all_fao_output.items():
    tmp = df.copy()
    tmp.index = pd.to_datetime(tmp.index)

    for col in ["Irrig", "Rain", "T", "E", "ETa", "DP", "Runoff", "Dr"]:
        tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    tmp["site"] = site
    daily_outputs.append(tmp.reset_index(names="date"))

daily_outputs = pd.concat(daily_outputs, ignore_index=True)

# Irrigation events only
irr_events = daily_outputs.loc[daily_outputs["Irrig"] > 0].copy()

fig, ax = plt.subplots(figsize=(9, 4))

irr_events.boxplot(
    column="Irrig",
    by="site",
    ax=ax,
    rot=45,
)

ax.grid(False)
ax.set(ylabel="Irrigation per event, mm", title='', xlabel='')
fig.suptitle('')
plt.tight_layout()

## Compare annual water supply with transpiration and total ET

In [ ]:
daily_outputs["year"] = daily_outputs["date"].dt.year

annual = (
    daily_outputs
    .groupby(["site", "year"])[["Rain", "Irrig", "T", "E", "ETa", "DP", "Runoff"]]
    .sum()
    .reset_index()
)

annual["water_input"] = annual["Rain"] + annual["Irrig"]

fig, ax = plt.subplots(figsize=(6, 6))

for site, grp in annual.groupby("site"):
    ax.scatter(
        grp["water_input"],
        grp["ETa"],
        label=site,
        alpha=0.7,
    )

limit = max(
    annual["water_input"].max(),
    annual["ETa"].max(),
) * 1.05

ax.plot([0, limit], [0, limit], linestyle="--", color='k')
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)

ax.set_xlabel("Annual precipitation + irrigation, mm/year")
ax.set_ylabel("Annual actual ET, mm/year")
ax.set_title("Annual water input versus actual ET")
ax.legend(fontsize=8)

plt.tight_layout()

## Same as above, but only transpiration

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for site, grp in annual.groupby("site"):
    ax.scatter(
        grp["water_input"],
        grp["T"],
        label=site,
        alpha=0.7,
    )

limit = max(
    annual["water_input"].max(),
    annual["T"].max(),
) * 1.05

ax.plot([0, limit], [0, limit], linestyle="--", color='k')
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)

ax.set_xlabel("Annual precipitation + irrigation, mm/year")
ax.set_ylabel("Annual transpiration, mm/year")
ax.set_title("Annual water input versus crop transpiration")

plt.tight_layout()

## Compare constant $p$ vs variable $p$ for one site

$p$ is the average fraction of Total Available Soil Water (TAW) that can be depleted from the root zone before moisture stress (reduction in ET). We can either set this to a constant based on Table 22 in FAO-56, or it can vary with $ET_c$ (see p. 162 in FAO-56).

In [ ]:
site = 'Pullman'
year = 2011

# First, recalculate with cons_p=False
start_date = pd.Timestamp(f"{year}-01-01").strftime("%Y-%j")
end_date = pd.Timestamp(f"{year}-12-31").strftime("%Y-%j")

# Get weather, parameter, update and irrigation objects
wth = wth_obj_all_sites[site][year]
par = par_objects[year]
upd = upd_objects[year]
autoirr = irr_objects[year]
mdl = fao.Model(start_date, end_date, par, wth, upd=upd, autoirr=autoirr, cons_p=False)
mdl.run()

new_output = mdl.odata.copy()
new_output.index = pd.to_datetime(new_output.index, format="%Y-%j")

# Get original output
fao_output = all_fao_output[site]

sub_df = fao_output.loc[f'{year}-01-01':f'{year}-12-31', :]
fig, ax = plt.subplots(2, figsize=(10, 6), sharex=True, tight_layout=True)

# Plot original evapotranspiration
ax[0].plot(sub_df.index, sub_df['E'], color='saddlebrown', label='Constant $p$')
ax[0].plot(sub_df.index, sub_df['T'], color='forestgreen')
ax[0].plot(new_output.index, new_output['E'], color='saddlebrown', ls='--', label='Variable $p$')
ax[0].plot(new_output.index, new_output['T'], color='forestgreen', ls='--')

# Add in precipitation and irrigation as stacked bar chart from the top
ax[1].bar(sub_df.index, sub_df['Irrig'], color='firebrick', label='Constant $p$')
ax[1].bar(new_output.index+pd.Timedelta(days=2), new_output['Irrig'], color='steelblue', label='Variable $p$')

ax[0].set(ylabel='Daily evapotranspiration (mm)', title=f'Comparison of different $p$ values for {site}')
ax[1].set(ylabel='Daily irrigation (mm)', xlim=[sub_df.index[0], sub_df.index[-1]])
ax[0].legend()
ax[1].legend()

## Compare impact of management allowed depletion values

Management allowed depletion is the fraction of total available water that can be depleted from the root zone before irrigation is triggered. The default value for corn is 50%, but let's also test 30% and 70%.

In [ ]:
site = 'Pullman'
year = 2011

# First, plot original values with MAD=0.5
# Get original output
fao_output = all_fao_output[site]
sub_df = fao_output.loc[f'{year}-01-01':f'{year}-12-31', :]
fig, ax = plt.subplots(2, figsize=(10, 6), sharex=True, tight_layout=True)

ax[0].plot(sub_df.index, sub_df['E'], color='saddlebrown', label=f'`mad`=0.5')
ax[0].plot(sub_df.index, sub_df['T'], color='forestgreen')
ax[1].bar(sub_df.index, sub_df['Irrig'], color='firebrick', label='`mad`=0.5')
ax[1].text(0.02, 0.98, f'Total irrigation for `mad`=0.5: {sub_df["Irrig"].sum():.0f} mm', transform=ax[1].transAxes, va="top", ha="left")

ls = ['--', ':']
c = ['lightcoral', 'orange']
for i, mad in enumerate([0.3, 0.7]):

    # Next, recalculate with new mad value
    start_date = pd.Timestamp(f"{year}-01-01").strftime("%Y-%j")
    end_date = pd.Timestamp(f"{year}-12-31").strftime("%Y-%j")

    # Get weather, parameter, update and irrigation objects
    wth = wth_obj_all_sites[site][year]
    par = par_objects[year]
    upd = upd_objects[year]
    planting_date = pd.to_datetime(f'{year}-{planting_day}')
    growing_length = sum(growth_stages.values())
    harvest_date = planting_date + pd.Timedelta(days=growing_length)
    autoirr = fao.AutoIrrigate()

    autoirr.addset(planting_date.strftime("%Y-%j"),
                   harvest_date.strftime("%Y-%j"),
                   mad=mad)
    mdl = fao.Model(start_date, end_date, par, wth, upd=upd, autoirr=autoirr, cons_p=True)
    mdl.run()

    new_output = mdl.odata.copy()
    new_output.index = pd.to_datetime(new_output.index, format="%Y-%j")

    ax[0].plot(new_output.index, new_output['E'], color='saddlebrown', ls=ls[i], label=f'`mad`={mad}')
    ax[0].plot(new_output.index, new_output['T'], color='forestgreen', ls=ls[i])

    # Add in precipitation and irrigation as stacked bar chart from the top
    ax[1].bar(new_output.index+pd.Timedelta(days=2*i), new_output['Irrig'], color=c[i], label=f'`mad`={mad}')

    # Add total irrigation to plot
    ax[1].text(0.02, 0.88-i*0.1, f'Total irrigation for `mad`={mad}: {new_output["Irrig"].sum():.0f} mm', transform=ax[1].transAxes, va="top", ha="left")

ax[0].set(ylabel='Daily evapotranspiration (mm)', title=site)
ax[1].set(ylabel='Daily irrigation (mm)', xlim=[sub_df.index[0], sub_df.index[-1]])
ax[0].legend()
ax[1].legend()